# H&M 적응형 M2 단독 · 마지막 주 seed49 첫 계산 배치

2020-09-15까지 학습, 09-16~22 test. seed49를 고정300epoch로 한 번만 평가합니다. **최종 10시드 평균이나 성공 판정이 아닙니다.** 같은 마지막 주를 보고 모형·계수·epoch를 바꾸지 마세요. H&M은 데이터·그래프가 커서 한 시드도 오래 걸릴 수 있고 GPU/메모리 여유가 필요합니다. 중단되면 실행 셀을 다시 눌러 재개하세요.

In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
SOURCE_COMMIT = '9891950363d9ea97e5d6a74df9176d12b6058d88'
REPO = Path('/content/clv-m2-adaptive-final-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','-q','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','-q','origin',SOURCE_COMMIT], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q','--detach',SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', torch.cuda.get_device_name(0), '| code:', SOURCE_COMMIT[:12])

In [ ]:
import clv_m2_adaptive_lastweek_final10 as screen
cfg = screen.configure('hm')
print(json.dumps(screen.preflight(cfg), ensure_ascii=False, indent=2))
prepared = screen.prepare(cfg)
print('분할:', prepared['data']['data_stats']['split_boundaries'])
print('데이터별 초기값:', prepared['protocol']['dataset_initialization'])

In [ ]:
result = screen.run_batch(prepared, cfg)
assert not result['complete'] and result['completed_seeds'] == [49]
print('H&M seed49 부분 결과. 최종 우열·유의성은 미판정입니다.')

In [ ]:
from IPython.display import display
columns = ['seed','recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50','price_purchase_amount_weighted_hit@10','vndcg@10']
display(result['absolute'][columns])
print(result['paths'])

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
output = Path('/content/clv_m2_adaptive_hm_seed49_lastweek_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    archive.write(prepared['run_dir'] / 'protocol.json', 'protocol.json')
    for name, path in result['paths'].items():
        archive.write(path, Path(path).name)
files.download(str(output))